<a href="https://colab.research.google.com/github/Pazidu/Research-Project/blob/main/EfficientNetV2S_HAM10000_RAM_SAFE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/drive')

Drive already mounted at /drive; to attempt to forcibly remount, call drive.mount("/drive", force_remount=True).


In [2]:
import os
import random
import shutil
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers, Model, mixed_precision
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)
from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

In [3]:
gpus = tf.config.list_physical_devices("GPU")

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

In [4]:
mixed_precision.set_global_policy("mixed_float16")

In [5]:
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

In [6]:
DATASET = "/content/newdata"
IMG_SRC = "/drive/MyDrive/Colab Notebooks/newdata"
CHECKPOINT = "/drive/MyDrive/checkpoints/best_model.keras"
MODEL_SAVE = "/drive/MyDrive/Colab Notebooks/Models/dermoscopy/final_model.keras"

In [11]:
if os.path.exists(DATASET):
    shutil.rmtree(DATASET)


# copy dataset from Drive

shutil.copytree(
    IMG_SRC,
    DATASET
)

'/content/newdata'

In [7]:
os.makedirs(
    "/drive/MyDrive/checkpoints",
    exist_ok=True
)

In [8]:
os.makedirs(
    "/drive/MyDrive/Colab Notebooks/Models/dermoscopy",
    exist_ok=True
)


print("Dataset copied successfully")
print("Dataset location:", DATASET)

Dataset copied successfully
Dataset location: /content/newdata


In [9]:
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS_STAGE1 = 20
EPOCHS_STAGE2 = 10

AUTOTUNE = 2

In [10]:
augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10)
])

In [11]:
def add_edge(image, label):
    image = tf.cast(image, tf.float32)
    gray = tf.image.rgb_to_grayscale(image)
    sobel = tf.image.sobel_edges(gray)
    edge = tf.sqrt(
        tf.reduce_sum(
            tf.square(sobel),
            axis=-1
        )
    )
    edge = tf.clip_by_value(edge, 0.0, 1.0)
    return (image, edge), label

In [12]:
def load_dataset(path, shuffle):
    ds = tf.keras.utils.image_dataset_from_directory(
        path,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        label_mode="categorical",
        shuffle=shuffle,
        seed=SEED
    )

    ds = ds.map(add_edge,num_parallel_calls=AUTOTUNE)
    ds = ds.prefetch(1)
    return ds

In [13]:
train_ds = load_dataset(DATASET + "/train",True)
val_ds = load_dataset(DATASET + "/valid",False)
test_ds = load_dataset(DATASET + "/test",False)

print("\nDatasets Loaded Successfully")

Found 8012 files belonging to 2 classes.
Found 1001 files belonging to 2 classes.
Found 1002 files belonging to 2 classes.

Datasets Loaded Successfully


In [14]:
def create_model():

    rgb_input = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3),name="rgb")
    edge_input = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1),name="edge")

    # -------------------------
    # RGB BRANCH
    # -------------------------

    x = augmentation(rgb_input)
    x = preprocess_input(x)
    backbone = EfficientNetV2S(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    # Stage 1:
    # Freeze backbone to save memory

    backbone.trainable = False
    features = backbone(x)

    # -------------------------
    # EDGE BRANCH
    # -------------------------

    e = layers.Conv2D(32,3,padding="same",activation="relu")(edge_input)
    e = layers.BatchNormalization()(e)
    e = layers.MaxPooling2D()(e)
    e = layers.Conv2D(64,3,padding="same",activation="relu")(e)
    e = layers.BatchNormalization()(e)
    e = layers.MaxPooling2D()(e)
    e = layers.Conv2D(128,3,padding="same",activation="relu")(e)

    # Resize edge features to EfficientNet output

    e = layers.Resizing(features.shape[1],features.shape[2])(e)
    e = layers.Conv2D(features.shape[-1],1,padding="same")(e)

    # -------------------------
    # FUSION
    # -------------------------

    fused = layers.Concatenate()([features,e])
    fused = layers.GlobalAveragePooling2D()(fused)
    fused = layers.Dense(256,activation="relu")(fused)
    fused = layers.Dropout(0.5)(fused)
    output = layers.Dense(2,activation="softmax",dtype="float32")(fused)
    model = Model(
        inputs=[
            rgb_input,
            edge_input
        ],
        outputs=output
    )


    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="categorical_crossentropy",

        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(
                name="auc"
            ),

            tf.keras.metrics.Precision(
                name="precision"
            ),

            tf.keras.metrics.Recall(
                name="recall"
            )
        ]
    )
    return model,backbone

In [15]:
model, backbone = create_model()
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ edge (InputLayer)   │ (None, 256, 256,  │          0 │ -                 │
│                     │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 256, 256,  │        320 │ edge[0][0]        │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256, 256,  │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 128, 128,  │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rgb (InputLayer)    │ (None, 256, 256,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 64,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 256, 256,  │          0 │ rgb[0][0]         │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resizing (Resizing) │ (None, 8, 8, 128) │          0 │ conv2d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ efficientnetv2-s    │ (None, 8, 8,      │ 20,331,360 │ sequential[0][0]  │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 8, 8,      │    165,120 │ resizing[0][0]    │
│                     │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 8, 8,      │          0 │ efficientnetv2-s… │
│ (Concatenate)       │ 2560)             │            │ conv2d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2560)      │          0 │ concatenate[0][0] │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    655,616 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 2)         │        514 │ dropout[0][0]   

 Total params: 21,245,666 (81.05 MB)

 Trainable params: 914,114 (3.49 MB)

 Non-trainable params: 20,331,552 (77.56 MB)

In [16]:
checkpoint = ModelCheckpoint(
    filepath=CHECKPOINT,
    monitor="val_auc",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_auc",
    patience=6,
    mode="max",
    restore_best_weights=True,
    verbose=1
)

lr_reduce = ReduceLROnPlateau(
    monitor="val_auc",
    factor=0.5,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

callbacks = [
    checkpoint,
    early_stop,
    lr_reduce
]

In [17]:
print("\n==============================")
print("STAGE 1 TRAINING")
print("==============================")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)


STAGE 1 TRAINING
Epoch 1/20
1002/1002 ━━━━━━━━━━━━━━━━━━━━ 0s 157ms/step - accuracy: 0.8712 - auc: 0.9227 - loss: 0.3430 - precision: 0.8712 - recall: 0.8712
Epoch 1: val_auc improved from None to 0.94938, saving model to /drive/MyDrive/checkpoints/best_model.keras

Epoch 1: finished saving model to /drive/MyDrive/checkpoints/best_model.keras
1002/1002 ━━━━━━━━━━━━━━━━━━━━ 234s 187ms/step - accuracy: 0.8850 - auc: 0.9398 - loss: 0.3081 - precision: 0.8850 - recall: 0.8850 - val_accuracy: 0.8911 - val_auc: 0.9494 - val_loss: 0.2855 - val_precision: 0.8911 - val_recall: 0.8911 - learning_rate: 1.0000e-04
Epoch 2/20
1002/1002 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step - accuracy: 0.8942 - auc: 0.9502 - loss: 0.2825 - precision: 0.8942 - recall: 0.8942
Epoch 2: val_auc improved from 0.94938 to 0.95219, saving model to /drive/MyDrive/checkpoints/best_model.keras

Epoch 2: finished saving model to /drive/MyDrive/checkpoints/best_model.keras
1002/1002 ━━━━━━━━━━━━━━━━━━━━ 166s 166ms/step - accuracy:

In [18]:
print("\n==============================")
print("STAGE 2 FINE TUNING")
print("==============================")


STAGE 2 FINE TUNING


In [19]:
backbone.trainable = True

for layer in backbone.layers[:-50]:
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

history2 = model.fit(train_ds,validation_data=val_ds,epochs=10,callbacks=callbacks)

Epoch 1/10
1002/1002 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step - accuracy: 0.8995 - auc: 0.9658 - loss: 0.2368 - precision: 0.8995 - recall: 0.8995
Epoch 1: val_auc did not improve from 0.95959
1002/1002 ━━━━━━━━━━━━━━━━━━━━ 229s 189ms/step - accuracy: 0.9025 - auc: 0.9675 - loss: 0.2307 - precision: 0.9025 - recall: 0.9025 - val_accuracy: 0.9001 - val_auc: 0.9559 - val_loss: 0.2774 - val_precision: 0.9001 - val_recall: 0.9001 - learning_rate: 1.0000e-05
Epoch 2/10
1002/1002 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step - accuracy: 0.9103 - auc: 0.9710 - loss: 0.2178 - precision: 0.9103 - recall: 0.9103
Epoch 2: val_auc did not improve from 0.95959

Epoch 2: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-06.
1002/1002 ━━━━━━━━━━━━━━━━━━━━ 172s 172ms/step - accuracy: 0.9133 - auc: 0.9712 - loss: 0.2171 - precision: 0.9133 - recall: 0.9133 - val_accuracy: 0.8971 - val_auc: 0.9574 - val_loss: 0.2662 - val_precision: 0.8971 - val_recall: 0.8971 - learning_rate: 1.0000e-05
Epoch 3/10
1002/100

In [20]:
print("\nLoading best checkpoint...")
model.load_weights(CHECKPOINT)


Loading best checkpoint...


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'loss_scale_optimizer', because it has 122 variables whereas the saved optimizer has 38 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 118 variables whereas the saved optimizer has 34 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [21]:
print("\n==============================")
print("FINAL TEST RESULTS")
print("==============================")

results = model.evaluate(test_ds)
print("Loss, Accuracy, AUC, Precision, Recall:")
print(results)


FINAL TEST RESULTS
126/126 ━━━━━━━━━━━━━━━━━━━━ 22s 176ms/step - accuracy: 0.8922 - auc: 0.9591 - loss: 0.2572 - precision: 0.8922 - recall: 0.8922
Loss, Accuracy, AUC, Precision, Recall:
[0.2571888267993927, 0.8922155499458313, 0.9590942859649658, 0.8922155499458313, 0.8922155499458313]


In [22]:
y_true = []
y_pred = []

for images, labels in test_ds:
    predictions = model.predict(images,verbose=0)
    y_true.extend(np.argmax(labels.numpy(),axis=1))
    y_pred.extend(np.argmax(predictions,axis=1))

cm = confusion_matrix(y_true,y_pred)
print("\nConfusion Matrix")
print(cm)
print("\nClassification Report")
print(classification_report(
        y_true,
        y_pred,
        target_names=[
            "melanoma",
            "non_melanoma"
        ]
    )
)


Confusion Matrix
[[ 24  88]
 [ 20 870]]

Classification Report
              precision    recall  f1-score   support

    melanoma       0.55      0.21      0.31       112
non_melanoma       0.91      0.98      0.94       890

    accuracy                           0.89      1002
   macro avg       0.73      0.60      0.62      1002
weighted avg       0.87      0.89      0.87      1002



In [24]:
model.save(MODEL_SAVE)
print("\nMODEL SAVED SUCCESSFULLY")


MODEL SAVED SUCCESSFULLY
